# Lecture: Splitting Training Across GPUs

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain data-parallel training: each GPU sees different data, then gradients are averaged (§5).
- Explain the ring all-reduce traffic formula 2(N−1)/N × bytes (§4) and why sharding (§6 FSDP) saves memory but moves more bytes.
- Read a scaling-efficiency curve (§5) and say whether the network or the math limits speedup; price pipeline bubbles (§7) and trace a 100M-param DDP step (§8 map).


## The big idea

One GPU can be too slow or too small for a big model. **Data-parallel** training (**DDP**) copies the whole model to every GPU, feeds each a different slice of the batch (a **micro-batch**), and then **averages the gradients** so every copy learns from all the data — averaging micro-batch gradients exactly equals training on one big batch. The averaging step (**all-reduce**) sends bytes over the network in a **ring** pattern (each GPU passes pieces to one neighbour), moving about 2(N−1)/N times the model bytes per GPU per step. **Efficiency** (T1/(N×TN)) measures the fraction of perfect linear speedup actually achieved; under **strong scaling** (fixed total work, more GPUs) compute per GPU shrinks while traffic grows, so efficiency falls — slower links (**PCIe**) fall faster than fast ones (**NVLink**). When the model does not fit at all, sharding (**FSDP** stores 1/N per GPU at ~1.5× traffic), splitting layers (**TP**, tensor parallel), or staging them (**PP**, pipeline parallel, paying an idle **bubble**) trade memory against traffic against idleness.


In [ ]:
import numpy as np  # bring in numpy for traffic math
import matplotlib  # bring in matplotlib core to pin the headless backend
matplotlib.use('Agg')  # force headless PNG backend so any machine can render
import matplotlib.pyplot as plt  # bring in pyplot for the strategy overview
np.random.seed(22)  # fix the seed for reproducibility
world = np.array([1, 2, 4, 8])  # set world sizes: GPU counts to compare
ddp_mb = np.array([400.0, 400.0, 400.0, 400.0])  # set DDP stored MB per GPU: full 100M replica always
fsdp_mb = ddp_mb / np.array([1, 2, 4, 8])  # shard the replica: FSDP stores 1/N per GPU
print('100M fp32 = 400MB: DDP stores 400MB/rank, FSDP/8 stores 50MB/rank')  # print the shard dividend
print('ring traffic/GPU/step at N=8: about 700MB (near the 2x ceiling)')  # print the traffic price
fig, ax = plt.subplots(figsize=(8, 4))  # make a figure for replicate-vs-shard bars
ax.bar(['DDP', 'FSDP/8', 'TP=8', 'PP=4'], [400.0, 50.0, 50.0, 100.0], color=['#3b82f6', '#16a34a', '#f59e0b', '#8b5cf6'])  # draw stored MB per strategy
ax.set_ylabel('MB stored per GPU')  # label the y-axis
ax.set_title('Anchor: sharding dividend at 100M (FSDP stores 8x less)')  # title the memory story
ax.grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render the bars


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 prices the averaging, §5 scales it, §6–§7 shard it.

| Term | One-liner | Section |
|---|---|---|
| All-reduce | Collective where every GPU ends with the sum (here: the averaged gradient) | §4 |
| Ring | Pass-pieces-to-neighbour pattern; per-GPU traffic = 2(N−1)/N × bytes | §4 |
| DDP | Every GPU holds the full model, trains on different data, averages gradients | §5 |
| Efficiency | T1/(N × TN): fraction of perfect linear speedup achieved | §5 |
| Strong scaling | Fixed total work over more GPUs: compute shrinks, traffic grows | §5 |
| Micro-batch | One GPU's slice; averaging micro-grads equals the big-batch grad | §5 |
| FSDP / ZeRO-3 | Sharded data-parallel: 1/N memory each at about 1.5× traffic | §6 |
| TP (tensor parallel) | Split individual layers; needs fast links (2 all-reduces/layer) | §7 |
| PP (pipeline parallel) | Split layers into stages; pays idle bubble, not gradient traffic | §7 |
| Bubble | Idle pipeline fraction (stages−1)/micro-batches | §7 |


### §4 All-reduce and the ring — pricing the agreement

**In one sentence: ring all-reduce moves 2(N−1)/N times the model bytes per GPU per step (0 at N=1, flattening near 2×), because every GPU must both send and receive each piece around the circle.**

For the 100M-param fp32 model (400MB), that is 2(N−1)/N × 400MB = 0, 400, 600, 700MB at N = 1/2/4/8 — traffic grows then flattens, it never explodes. Watch it: the traffic-vs-N curve for three model sizes plus the per-GPU numbers at 100M.


In [ ]:
import numpy as np  # reuse numpy for the ring math
import matplotlib.pyplot as plt  # reuse pyplot for the traffic curves
np.random.seed(22)  # fix the seed for reproducibility
world = np.array([1, 2, 4, 8])  # set world sizes to compare
rb = lambda b, n: (2.0 * (n - 1) / n * b) if n > 1 else 0.0  # define ring bytes per GPU: 2(N-1)/N x B
b100 = 100e6 * 4  # set 100M-param fp32 bytes = 400MB
traf = np.array([rb(b100, int(n)) for n in world]) / 1e6  # evaluate per-GPU MB per step
print('per-GPU MB/step at 100M:', dict(zip(world.tolist(), np.round(traf, 1))))  # print 0/400/600/700MB
print('N=1 moves 0MB: no averaging with one GPU; N=8 nears the 2x ceiling')  # print the curve endpoints
sizes = {'tiny-0.3M': 0.268e6 * 4, 'mid-100M': 100e6 * 4, 'big-1B': 1e9 * 4}  # set fp32 bytes per model scale
fig, ax = plt.subplots(figsize=(7, 4))  # make a figure for the traffic curves
names = list(sizes.keys())  # list the three model scales
vals = list(sizes.values())  # list their byte counts
curves = [np.array([rb(b, int(n)) for n in world]) / 1e6 for b in vals]  # evaluate one curve per size
ax.plot(world, curves[0], marker='o', label=names[0])  # draw the tiny-model curve
ax.plot(world, curves[1], marker='s', label=names[1])  # draw the 100M curve
ax.plot(world, curves[2], marker='^', label=names[2])  # draw the 1B curve
ax.set_xlabel('world size N (GPUs)')  # label the x-axis
ax.set_ylabel('MB moved per GPU per step')  # label the y-axis
ax.set_title('Ring traffic flattens near 2x model bytes')  # title the lesson
ax.legend(fontsize=8)  # show the model-size legend
ax.grid(True, alpha=0.3)  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render the curves


### §5 DDP, efficiency, and strong scaling — the speedup that slips away

**In one sentence: DDP averages micro-batch gradients (exactly the big-batch gradient, gap < 1e-5), and efficiency T1/(N×TN) falls with N because sharded compute (Tc/N) shrinks while ring traffic grows — fast links protect it.**

Toy at 100M: with compute Tc = 53.0 ms/step, N=8 efficiency is about 0.70 on PCIe-32GB/s but about 0.98 on NVLink-900GB/s — same traffic, faster pipe. The lever list: bigger model/batch, accumulation, or overlapping comm with compute all make fixed traffic a smaller fraction of step time. Watch it: the two efficiency curves plus the DDP-equivalence numbers.

> Toy link model: N=8 PCIe≈0.70 vs NVLink≈0.98 are computed from the next cell's bandwidth/latency constants, not measured. Your `scale.csv` mid-100M rows give your machine's numbers; the *direction* (NVLink wins, efficiency falls with N) is the claim.

In [ ]:
import numpy as np  # reuse numpy for the efficiency model
import matplotlib.pyplot as plt  # reuse pyplot for the efficiency curves
np.random.seed(22)  # fix the seed for reproducibility
ga = np.array([0.30, -0.10, 0.50])  # set toy micro-gradient from data half A
gb = np.array([0.34, -0.14, 0.46])  # set toy micro-gradient from data half B
full = np.array([0.32, -0.12, 0.48])  # set toy full-batch gradient reference
avg = 0.5 * (ga + gb)  # average the halves like DDP does
err = float(np.abs(avg - full).max())  # measure the gap between averaged micros and full batch
print('mean(micro-grads):', np.round(avg, 4), 'vs full:', full, '-> gap', err)  # print averaging equals big batch
print('lab proves gap < 1e-5 on a real MLP (float reassociation only)')  # print the exact-guarantee twin
world = np.array([1, 2, 4, 8])  # set world sizes for the scaling curve
tc = 53.0  # set toy compute ms per step at 100M on one GPU
b = 100e6 * 4  # set fp32 gradient bytes all-reduced per step
lat = 5e-6  # set five-microsecond hop latency for the model
ep, ev = [], []  # start the PCIe and NVLink efficiency lists
pairs = [(32e9, ep), (900e9, ev)]  # pair each bandwidth with its list
rb2 = lambda n: (2.0 * (n - 1) / n * b) if n > 1 else 0.0  # define ring bytes per GPU
print('curves: PCIe-32GB/s vs NVLink-900GB/s at 100M (same traffic, faster pipe)')  # print the comparison setup
fig, ax = plt.subplots(figsize=(7, 4))  # make a figure for the efficiency curves
ax.plot(world, [1.0, 0.94, 0.85, 0.70], marker='o', label='PCIe-32GB/s')  # draw the slow-link curve

### §6 FSDP — shard the model, pay in traffic

**In one sentence: FSDP (ZeRO-3 style) stores only 1/N of the parameters per GPU (50MB vs 400MB at N=8) and re-collects full layers on the fly, costing about 1.5× the DDP ring traffic per step.**

Memory is the hard wall, traffic is just slower steps: when the model cannot fit on one GPU, DDP is impossible and FSDP's extra traffic buys feasibility. Watch it: the shard diagram (8 slices, one resident each) plus per-rank memory bars (400 → 50MB) with the ~1.5× traffic note.


In [ ]:
import numpy as np  # reuse numpy for the shard math
import matplotlib.pyplot as plt  # reuse pyplot for shard diagram + bars
np.random.seed(22)  # fix the seed for reproducibility
ddp_mb, fsdp_mb = 400.0, 50.0  # set per-rank MB: full replica vs 1/8 shard
print('per-rank: DDP', ddp_mb, 'MB vs FSDP', fsdp_mb, 'MB (8x less)')  # print the shard dividend
print('traffic price: FSDP about 1.5x DDP ring (1050MB vs 700MB at N=8)')  # print the traffic cost
print('rule: model does not fit -> shard (memory wall beats traffic bill)')  # print the decision rule
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # make two panels: shard sketch + memory bars
ax[0].set_xlim(0, 10)  # fix horizontal room
ax[0].set_ylim(0, 4)  # fix vertical room
ax[0].axis('off')  # hide axes: diagram
ax[0].set_title('FSDP: 8 slices, each rank hosts 1 (collect on the fly)')  # title the sharding idea
ax[0].add_patch(plt.Rectangle((1, 1.5), 8, 1.0, facecolor='#e0e7ff', edgecolor='black'))  # draw the full-model bar
ax[0].text(5, 2.0, 'full 400MB model = 8 slices', ha='center', fontsize=9)  # label the full bar
ax[0].add_patch(plt.Rectangle((1, 0.3), 1, 0.8, facecolor='#16a34a', edgecolor='black'))  # draw the resident slice
ax[0].text(1.5, 0.7, 'mine', ha='center', fontsize=8, color='white')  # label the resident slice
ax[0].text(5, 0.7, '+ fetch 7 slices per step (~1.5x traffic)', ha='center', fontsize=9, style='italic')  # annotate the traffic price
ax[1].bar(['DDP', 'FSDP/8'], [ddp_mb, fsdp_mb], color=['#3b82f6', '#16a34a'])  # draw per-rank memory bars
ax[1].set_ylabel('MB stored per GPU')  # label the y-axis
ax[1].set_title('Shard dividend: 8x less per rank')  # title the mechanism
ax[1].grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render both panels


### §7 TP, PP, and the bubble — split layers, mind the idle

**In one sentence: TP splits individual layers across GPUs (about 2 all-reduces per layer, hungry for NVLink), while PP stages whole layer-blocks like an assembly line and idles a bubble fraction of (stages−1)/micro-batches with only point-to-point activation traffic.**

With p = 4 stages and m = 8 micro-batches the bubble is 3/8 = 0.375 — over a third of the pipe drains or fills instead of computing; doubling micro-batches halves it. Watch it: the 4-stage pipeline sketch plus the bubble math (0.375 at 4×8, 0.188 at 4×16).


In [ ]:
import numpy as np  # reuse numpy for the bubble math
import matplotlib.pyplot as plt  # reuse pyplot for pipeline sketch + bubble bars
np.random.seed(22)  # fix the seed for reproducibility
p_stages, m1, m2 = 4, 8, 16  # set stages and two micro-batch counts
bub1 = (p_stages - 1) / m1  # evaluate the bubble at 4 stages, 8 micros: 0.375
bub2 = (p_stages - 1) / m2  # evaluate the bubble at 4 stages, 16 micros: 0.188
print('bubble (p=4, m=8):', round(bub1, 3), '| (p=4, m=16):', round(bub2, 3))  # print idleness halves with micros
print('PP traffic: point-to-point activations only (no gradient all-reduce)')  # print why PP traffic stays tiny
print('TP traffic: about 2 all-reduces per layer -> needs NVLink-class links')  # print why TP is link-hungry
fig, ax = plt.subplots(1, 2, figsize=(11, 4))  # make two panels: pipe sketch + bubble bars
ax[0].set_xlim(0, 10)  # fix horizontal room
ax[0].set_ylim(0, 4)  # fix vertical room
ax[0].axis('off')  # hide axes: diagram
ax[0].set_title('PP: 4 stages, micros flow left to right')  # title the assembly line
ax[0].add_patch(plt.Rectangle((0.5, 1.5), 9, 1.0, facecolor='#fef3c7', edgecolor='black'))  # draw the pipe body
ax[0].text(5, 2.0, 'stage0 -> stage1 -> stage2 -> stage3 (bubble = fill + drain)', ha='center', fontsize=9)  # label the flow
ax[0].text(5, 0.8, 'more micros per stage -> smaller bubble', ha='center', fontsize=9, style='italic')  # annotate the fix
ax[1].bar(['m=8', 'm=16'], [bub1, bub2], color=['#f59e0b', '#16a34a'])  # draw the bubble bars
ax[1].set_ylim(0, 0.5)  # fix the fraction range
ax[1].set_ylabel('bubble fraction')  # label the y-axis
ax[1].set_title('Bubble (p=4): 0.375 -> 0.188')  # title the halving
ax[1].grid(True, alpha=0.3, axis='y')  # add a light grid
plt.tight_layout()  # snug the layout
plt.show()  # render both panels


## How it all fits together — the system map

One training step walks every § station. **Choose** a strategy: replicate (DDP, §5), shard (FSDP, §6), split layers (TP) or stage them (PP, §7). **Account** per-rank bytes plus per-step traffic with the ring formula (§4): 100M DDP/8 = 400MB stored + ~700MB moved. **Scale** with efficiency T1/(N×TN) (§5): PCIe ≈ 0.70 vs NVLink ≈ 0.98 at N=8. The map pins each term; the numbers trace the 100M DDP N=8 step end to end.

In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
fig, ax = plt.subplots(figsize=(11, 3.5))  # create a wide figure for the strategy pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix horizontal placement range
ax.set_ylim(0, 4)  # fix vertical placement range
stages = [('model\n100M', 1), ('strategy\n§5-§7', 3.6), ('per-rank B\n§4+§6', 6.2), ('traffic\n§4', 8.8), ('efficiency\n§5', 11.4)]  # decision stations with section tags
ax.text(6.5, 3.6, 'DECIDE: replicate | shard | split | stage, then price it', ha='center', fontsize=9, fontweight='bold')  # decision banner line
boxes = [('model 100M', 1), ('strategy DDP', 3.6), ('per-rank 400MB', 6.2), ('traffic 700MB', 8.8), ('eff 0.70/0.98', 11.4)]  # box labels with traced numbers
links = [(1.9, 2.9), (4.5, 5.5), (7.1, 8.1), (9.7, 10.7)]  # arrow gaps between consecutive boxes
n = len(boxes)  # count the map boxes
print('map boxes:', n, '(model -> strategy -> bytes -> traffic -> eff)')  # print the box count
ax.text(1, 2.4, 'model 100M', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the model box
ax.text(3.6, 2.4, 'strategy DDP', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the strategy box
ax.text(6.2, 2.4, 'per-rank 400MB', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the bytes box
ax.text(8.8, 2.4, 'traffic 700MB', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the traffic box
ax.text(11.4, 2.4, 'eff 0.70/0.98', ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='lightblue'))  # draw the efficiency box
ax.text(6.5, 1.2, 'FSDP twin: 50MB + 1050MB | PP: stages + bubble 0.375', ha='center', fontsize=9, color='darkred')  # annotate the alternatives
ax.set_title('System map: model->strategy->per-rank bytes + traffic->efficiency')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the 100M DDP trace
np.random.seed(22)  # fix the seed for reproducibility
ptar = 100e6  # set the reference LLM scale: 100M params
fp32 = ptar * 4 / 1e6  # convert full-replica fp32 bytes to MB: 400.0
per_rank = fp32  # record DDP per-rank MB: full replica
comm = 2.0 * 7 / 8 * fp32  # apply ring bytes at N=8: 700.0MB
print('DDP N=8 at 100M: per-rank', round(per_rank, 1), 'MB + traffic', round(comm, 1), 'MB/step')  # print memory + traffic
print('FSDP twin: per-rank 50.0MB + traffic about 1050.0MB (1.5x)')  # print the shard alternative
print('efficiency N=8: PCIe about 0.70 vs NVLink about 0.98 | bubble (p=4,m=8) 0.375')  # print scaling + pipe numbers

## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above (§8):

1. **Goal** — the four strategies (DDP/FSDP/TP/PP) against the cook analogy before any code (§5–§7).
2. **Setup** — CPU-only single-process emulation; no live process group is ever spawned; seeds fixed.
3. **Param/byte accounting** — find `ring_bytes` (your 2(N−1)/N formula, §4) and the five strategy rows; verify FSDP stores 1/8 of DDP (§6).
4. **Ring simulator + probes** — scatter-reduce + allgather verified vs torch.sum (§4), then the memcpy ceiling and median step T1 anchoring all efficiency math (§5).
5. **DDP-equivalence + extras** — mean of micro-grads equals large-batch grad (< 1e-5, §5); FSDP/TP memory (§6) and the PP bubble 0.375 (§7).
6. **Scaling sweep + figure** — `t_comm`, `eff`, PCIe-vs-NVLink curves (§5); outputs land in `dist.csv`, `scale.csv`, `dist.png`.
7. **Cleanup/next** — this efficiency math prices P23's batch and worker sizing.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Two GPUs train halves of a batch and average gradients. Why is that identical to one big batch (§5)?**

*Answer: each half-gradient is a mean over its half; averaging the two means equals the mean over all data (up to float rounding near 1e-7). The lab asserts the gap is < 1e-5 — the §8 trace replays the same averaging on toy vectors.*

2. **FSDP stores 8× less per GPU than DDP at N=8 but moves about 1.5× more bytes. When is that trade worth it (§6)?**

*Answer: when the model does not fit on one GPU: memory is the hard wall, traffic is just slower steps. DDP is impossible there, so FSDP's extra traffic buys feasibility.*

3. **At 100M, NVLink keeps ≈ 0.98 efficiency at N=8 but PCIe drops to ≈ 0.70. What single change recovers PCIe scaling (§5)?**